# DP Audit Autoresearch — Reference Model & LiRA Attacks

**EECE 608 — Mohamad Faour**

This notebook runs advanced membership inference attacks that need GPU:
- **Reference model attack**: train a model on non-members, compare losses
- **LiRA (Likelihood Ratio Attack)**: train K shadow models, use loss distributions
- Standard scoring baselines for comparison

### Setup
1. Runtime → Change runtime type → **T4 GPU**
2. Click **Run All** (Runtime → Run all)

---
## 0. Clone repo and install

In [1]:
import os, sys, shutil
from pathlib import Path

# Clone repo onto Colab's machine (GPU server, not your laptop)
WORK = Path("/content/EECE_608")
if not WORK.exists():
    !git clone https://github.com/mohdfaour03/EECE_608.git /content/EECE_608

os.chdir(WORK)
!pip install -q opacus>=1.5 dp-accounting>=0.4 pyyaml
!pip install -e . -q

sys.path.insert(0, str(WORK / "src"))
sys.path.insert(0, str(WORK / "autoresearch"))

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("Ready.")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for dp-audit-tightness (pyproject.toml) ... done
GPU: NVIDIA L4
Ready.


## 1. Train target model and establish baseline

In [2]:
import prepare
import torch
import torch.nn.functional as F
import random
import time
import json
import statistics
import logging
import numpy as np
from pathlib import Path
from dp_audit_tightness.privacy.empirical import estimate_empirical_lower_bound

logging.basicConfig(level=logging.INFO, format='%(message)s')

# Train/load target model
model, bundle, device, record = prepare.load_model_and_data()
print(f"Target model: epsilon_upper={record.epsilon_upper_theory:.4f}")
print(f"Device: {device}")
print(f"Train set: {len(bundle.train_dataset)}, Eval set: {len(bundle.eval_dataset)}")

Target model: epsilon_upper=0.7709
Device: cuda
Train set: 57000, Eval set: 3000


In [3]:
# --- Helper functions used by all experiments ---

def collect_logits(mdl, dataset, indices, dev, batch_size=512):
    """Get model logits and labels for given indices."""
    logits_list, labels_list = [], []
    with torch.no_grad():
        for s in range(0, len(indices), batch_size):
            bi = indices[s:s+batch_size]
            imgs, lbls = zip(*(dataset[i] for i in bi))
            x = torch.stack(imgs).to(dev)
            y = torch.tensor(lbls, dtype=torch.long, device=dev)
            logits_list.append(mdl(x))
            labels_list.append(y)
    return torch.cat(logits_list), torch.cat(labels_list)


def evaluate(member_scores, nonmember_scores, label=""):
    """Run both point and conservative estimates, print results."""
    eps_upper = record.epsilon_upper_theory
    gap = statistics.fmean(member_scores) - statistics.fmean(nonmember_scores)

    est_point = estimate_empirical_lower_bound(
        member_scores=member_scores, nonmember_scores=nonmember_scores,
        delta=prepare.DELTA, align_event_to_score_direction=True,
        require_member_favoring=True, report_confidence_supported_lower_bound=False,
    )
    est_cons = estimate_empirical_lower_bound(
        member_scores=member_scores, nonmember_scores=nonmember_scores,
        delta=prepare.DELTA, align_event_to_score_direction=True,
        require_member_favoring=True, report_confidence_supported_lower_bound=True,
    )

    tr_point = est_point.epsilon_lower_empirical / eps_upper if eps_upper > 0 else 0
    tr_cons = est_cons.epsilon_lower_empirical / eps_upper if eps_upper > 0 else 0

    print(f"  {label:30s} | gap={gap:+.4f} | point: eps={est_point.epsilon_lower_empirical:.4f} tr={tr_point:.4f} | "
          f"cons: eps={est_cons.epsilon_lower_empirical:.4f} tr={tr_cons:.4f} | "
          f"tpr={est_point.selected_tpr} fpr={est_point.selected_fpr} | "
          f"n={len(member_scores)}+{len(nonmember_scores)}")

    return {
        "label": label, "gap": gap,
        "eps_point": est_point.epsilon_lower_empirical,
        "eps_cons": est_cons.epsilon_lower_empirical,
        "tr_point": tr_point, "tr_cons": tr_cons,
        "tpr": est_point.selected_tpr, "fpr": est_point.selected_fpr,
        "n_member": len(member_scores), "n_nonmember": len(nonmember_scores),
    }

print("Helpers loaded.")

Helpers loaded.


## 2. Baseline: standard scoring (no reference model)

In [4]:
BUDGET = 1024
SEEDS = [401, 402, 403, 404, 405]
results = []

score_fns = {
    "neg_loss": lambda lg, lb: (-F.cross_entropy(lg, lb, reduction="none")).cpu().tolist(),
    "max_prob": lambda lg, lb: torch.softmax(lg, 1).max(1).values.cpu().tolist(),
    "logit_margin": lambda lg, lb: (lambda t: t[:,0]-t[:,1])(torch.softmax(lg,1).topk(2,1).values).cpu().tolist(),
    "correct_logit": lambda lg, lb: lg.gather(1, lb.unsqueeze(1)).squeeze().cpu().tolist(),
}

print("=== Baseline: standard passive scoring ===")
for sname, sfn in score_fns.items():
    all_m, all_n = [], []
    for seed in SEEDS:
        rng = random.Random(seed)
        mi = rng.sample(range(len(bundle.train_dataset)), BUDGET // 2)
        ni = rng.sample(range(len(bundle.eval_dataset)), BUDGET // 2)
        ml, mlb = collect_logits(model, bundle.train_dataset, mi, device)
        nl, nlb = collect_logits(model, bundle.eval_dataset, ni, device)
        all_m.extend(sfn(ml, mlb))
        all_n.extend(sfn(nl, nlb))
    r = evaluate(all_m, all_n, label=sname)
    results.append(r)

=== Baseline: standard passive scoring ===
  neg_loss                       | gap=-0.0025 | point: eps=1.3734 tr=1.7817 | cons: eps=0.0000 tr=0.0000 | tpr=0.00078125 fpr=0.0 | n=2560+2560
  max_prob                       | gap=+0.0015 | point: eps=1.3734 tr=1.7817 | cons: eps=0.0000 tr=0.0000 | tpr=0.00078125 fpr=0.0 | n=2560+2560
  logit_margin                   | gap=+0.0004 | point: eps=1.3734 tr=1.7817 | cons: eps=0.0000 tr=0.0000 | tpr=0.00078125 fpr=0.0 | n=2560+2560
  correct_logit                  | gap=+0.0627 | point: eps=0.6672 tr=0.8656 | cons: eps=0.0000 tr=0.0000 | tpr=0.000390625 fpr=0.0 | n=2560+2560


## 3. Reference Model Attack

Train a reference model on non-member data only. For each example x:
- `score = reference_loss(x) - target_loss(x)`
- If x is a member, the target model knows it well (low loss) but the reference doesn't → high score
- If x is a non-member, both models are similar → score near 0

In [7]:
from dp_audit_tightness.training.dp_sgd import train_dp_sgd
from dp_audit_tightness.data.datasets import DatasetBundle
from dp_audit_tightness.utils.seeds import set_global_seed
from torch.utils.data import Subset, random_split

def train_reference_model(eval_dataset, seed=999, epochs=1):
    """Train a DP-SGD model on non-member (eval) data only."""
    n = len(eval_dataset)
    n_train = int(n * 0.8)
    n_held = n - n_train
    ref_train, ref_held = random_split(eval_dataset, [n_train, n_held],
                                       generator=torch.Generator().manual_seed(seed))

    ref_bundle = DatasetBundle(
        train_dataset=ref_train,
        eval_dataset=ref_held,
        input_dim=prepare.INPUT_DIM,
        num_classes=prepare.NUM_CLASSES,
        train_size=len(ref_train),
        eval_size=len(ref_held),
    )

    config = prepare._build_train_config(epochs=epochs, seed=seed)
    set_global_seed(seed)
    logger = logging.getLogger("ref_model")
    outcome = train_dp_sgd(config=config, logger=logger, dataset_bundle=ref_bundle)

    from dp_audit_tightness.models.io import load_model_for_inference
    ref_model = load_model_for_inference(config.model, outcome.checkpoint_path, device=device)
    return ref_model

print("Training reference model on non-member data...")
t0 = time.time()
ref_model = train_reference_model(bundle.eval_dataset, seed=999)
print(f"Reference model trained in {time.time()-t0:.1f}s")

Training reference model on non-member data...


/usr/local/lib/python3.12/dist-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


Reference model trained in 1.5s


In [8]:
print("\n=== Reference Model Attack ===")

for budget_per_seed in [128, 256, 512, 1024]:
    all_m, all_n = [], []
    for seed in SEEDS:
        rng = random.Random(seed)
        mi = rng.sample(range(len(bundle.train_dataset)), budget_per_seed // 2)
        ni = rng.sample(range(len(bundle.eval_dataset)), budget_per_seed // 2)

        # Target model logits
        t_ml, t_mlb = collect_logits(model, bundle.train_dataset, mi, device)
        t_nl, t_nlb = collect_logits(model, bundle.eval_dataset, ni, device)

        # Reference model logits
        r_ml, _ = collect_logits(ref_model, bundle.train_dataset, mi, device)
        r_nl, _ = collect_logits(ref_model, bundle.eval_dataset, ni, device)

        # Score = reference_loss - target_loss
        # Members: target knows them (low loss), reference doesn't (high loss) → positive score
        target_loss_m = F.cross_entropy(t_ml, t_mlb, reduction="none")
        ref_loss_m = F.cross_entropy(r_ml, t_mlb, reduction="none")
        target_loss_n = F.cross_entropy(t_nl, t_nlb, reduction="none")
        ref_loss_n = F.cross_entropy(r_nl, t_nlb, reduction="none")

        score_m = (ref_loss_m - target_loss_m).cpu().tolist()
        score_n = (ref_loss_n - target_loss_n).cpu().tolist()

        all_m.extend(score_m)
        all_n.extend(score_n)

    r = evaluate(all_m, all_n, label=f"ref_attack budget={budget_per_seed}")
    results.append(r)


=== Reference Model Attack ===
  ref_attack budget=128          | gap=+0.1273 | point: eps=1.3861 tr=1.7981 | cons: eps=0.0000 tr=0.0000 | tpr=0.05 fpr=0.0125 | n=320+320
  ref_attack budget=256          | gap=+0.0443 | point: eps=1.3847 tr=1.7963 | cons: eps=0.0000 tr=0.0000 | tpr=0.00625 fpr=0.0015625 | n=640+640
  ref_attack budget=512          | gap=-0.0159 | point: eps=0.6915 tr=0.8971 | cons: eps=0.0000 tr=0.0000 | tpr=0.00625 fpr=0.003125 | n=1280+1280
  ref_attack budget=1024         | gap=+0.0077 | point: eps=1.3734 tr=1.7817 | cons: eps=0.0000 tr=0.0000 | tpr=0.00078125 fpr=0.0 | n=2560+2560


## 4. LiRA — Likelihood Ratio Attack

Train K shadow models on random 50% subsets of the training data.
For each target example x, compute loss under "IN" models (that included x)
and "OUT" models (that excluded x). The log-likelihood ratio is the score.

In [9]:
K_SHADOW = 8  # number of shadow models

shadow_models = []
shadow_member_sets = []

n_train = len(bundle.train_dataset)
half = n_train // 2

print(f"Training {K_SHADOW} shadow models (50% of training data each)...")
for k in range(K_SHADOW):
    t0 = time.time()
    shadow_seed = 5000 + k
    rng_k = random.Random(shadow_seed)

    all_indices = list(range(n_train))
    rng_k.shuffle(all_indices)
    in_indices = set(all_indices[:half])
    shadow_member_sets.append(in_indices)

    subset = Subset(bundle.train_dataset, list(in_indices))
    shadow_bundle = DatasetBundle(
        train_dataset=subset,
        eval_dataset=bundle.eval_dataset,
        input_dim=prepare.INPUT_DIM,
        num_classes=prepare.NUM_CLASSES,
        train_size=len(subset),
        eval_size=len(bundle.eval_dataset),
    )

    config = prepare._build_train_config(epochs=1, seed=shadow_seed)
    set_global_seed(shadow_seed)
    logger = logging.getLogger(f"shadow_{k}")
    outcome = train_dp_sgd(config=config, logger=logger, dataset_bundle=shadow_bundle)

    from dp_audit_tightness.models.io import load_model_for_inference
    sm = load_model_for_inference(config.model, outcome.checkpoint_path, device=device)
    shadow_models.append(sm)
    print(f"  Shadow {k}: {time.time()-t0:.1f}s")

print(f"Done. {K_SHADOW} shadow models trained.")

Training 8 shadow models (50% of training data each)...


  Shadow 0: 6.7s


  Shadow 1: 6.8s


  Shadow 2: 6.7s


  Shadow 3: 6.7s


  Shadow 4: 6.6s


  Shadow 5: 6.7s


  Shadow 6: 6.7s


  Shadow 7: 6.7s
Done. 8 shadow models trained.


In [10]:
print("\n=== LiRA Attack ===")

for budget_per_seed in [128, 256, 512]:
    all_m, all_n = [], []

    for seed in SEEDS:
        rng = random.Random(seed)
        mi = rng.sample(range(len(bundle.train_dataset)), budget_per_seed // 2)
        ni = rng.sample(range(len(bundle.eval_dataset)), budget_per_seed // 2)

        # For each member example, compute average loss under IN vs OUT shadow models
        for idx in mi:
            img, lbl = bundle.train_dataset[idx]
            x = img.unsqueeze(0).to(device)
            y = torch.tensor([lbl], dtype=torch.long, device=device)

            in_losses, out_losses = [], []
            for k, sm in enumerate(shadow_models):
                with torch.no_grad():
                    loss_k = F.cross_entropy(sm(x), y).item()
                if idx in shadow_member_sets[k]:
                    in_losses.append(loss_k)
                else:
                    out_losses.append(loss_k)

            # Score = mean(OUT losses) - mean(IN losses)
            # Members have lower IN loss → higher score
            if in_losses and out_losses:
                score = statistics.fmean(out_losses) - statistics.fmean(in_losses)
            elif out_losses:
                score = statistics.fmean(out_losses)
            else:
                score = -statistics.fmean(in_losses) if in_losses else 0.0
            all_m.append(score)

        # For non-members: all shadow models are "OUT" (they weren't in any training set)
        for idx in ni:
            img, lbl = bundle.eval_dataset[idx]
            x = img.unsqueeze(0).to(device)
            y = torch.tensor([lbl], dtype=torch.long, device=device)

            losses = []
            for sm in shadow_models:
                with torch.no_grad():
                    losses.append(F.cross_entropy(sm(x), y).item())

            # Non-members: use negative mean loss as baseline score
            # (no IN/OUT split — use target model loss as reference)
            with torch.no_grad():
                target_loss = F.cross_entropy(model(x), y).item()
            shadow_mean = statistics.fmean(losses)
            score = shadow_mean - target_loss
            all_n.append(score)

    r = evaluate(all_m, all_n, label=f"LiRA K={K_SHADOW} budget={budget_per_seed}")
    results.append(r)


=== LiRA Attack ===
  LiRA K=8 budget=128            | gap=-0.2080 | point: eps=0.0415 tr=0.0538 | cons: eps=0.0122 tr=0.0159 | tpr=1.0 fpr=0.959375 | n=320+320
  LiRA K=8 budget=256            | gap=-0.2407 | point: eps=0.6867 tr=0.8909 | cons: eps=0.0000 tr=0.0000 | tpr=0.0015625 fpr=0.0 | n=640+640
  LiRA K=8 budget=512            | gap=-0.2529 | point: eps=0.6803 tr=0.8825 | cons: eps=0.0000 tr=0.0000 | tpr=0.00078125 fpr=0.0 | n=1280+1280


## 5. Results Summary

In [11]:
import pandas as pd

df = pd.DataFrame(results)
df = df.sort_values("tr_point", ascending=False)

print("=" * 100)
print("FULL RESULTS — sorted by point-estimate tightness ratio")
print("=" * 100)
print(df[["label", "gap", "eps_point", "tr_point", "eps_cons", "tr_cons",
          "tpr", "fpr", "n_member"]].to_string(index=False))

print(f"\n--- Best point-estimate: {df.iloc[0]['label']} (tr={df.iloc[0]['tr_point']:.4f}) ---")
print(f"--- Best conservative:  ", end="")
cons_best = df[df["tr_cons"] > 0]
if len(cons_best) > 0:
    best_c = cons_best.sort_values("tr_cons", ascending=False).iloc[0]
    print(f"{best_c['label']} (tr={best_c['tr_cons']:.4f}) ---")
else:
    print("none (all conservative estimates are 0) ---")

# Save
out_dir = WORK / "autoresearch" / "results"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "experiment_results.csv", index=False)
print(f"\nSaved to {out_dir / 'experiment_results.csv'}")

FULL RESULTS — sorted by point-estimate tightness ratio
                 label       gap  eps_point  tr_point  eps_cons  tr_cons      tpr      fpr  n_member
 ref_attack budget=128  0.127278   1.386094  1.798127  0.000000 0.000000 0.050000 0.012500       320
 ref_attack budget=256  0.044272   1.384693  1.796309  0.000000 0.000000 0.006250 0.001563       640
              neg_loss -0.002497   1.373412  1.781674  0.000000 0.000000 0.000781 0.000000      2560
          logit_margin  0.000388   1.373412  1.781674  0.000000 0.000000 0.000781 0.000000      2560
              max_prob  0.001523   1.373412  1.781674  0.000000 0.000000 0.000781 0.000000      2560
ref_attack budget=1024  0.007689   1.373412  1.781674  0.000000 0.000000 0.000781 0.000000      2560
 ref_attack budget=512 -0.015874   0.691546  0.897116  0.000000 0.000000 0.006250 0.003125      1280
   LiRA K=8 budget=256 -0.240691   0.686727  0.890864  0.000000 0.000000 0.001563 0.000000       640
   LiRA K=8 budget=512 -0.252927   

## 6. Download results

In [12]:
from google.colab import files
files.download(str(out_dir / "experiment_results.csv"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
!cat /content/EECE_608/autoresearch/results/experiment_results.csv


label,gap,eps_point,eps_cons,tr_point,tr_cons,tpr,fpr,n_member,n_nonmember
ref_attack budget=128,0.12727793194353576,1.3860943411172235,0.0,1.7981271583061398,0.0,0.05,0.0125,320,320
ref_attack budget=256,0.04427172532305135,1.3846930797529169,0.0,1.7963093555489236,0.0,0.00625,0.0015625,640,640
neg_loss,-0.0024970004398640055,1.3734117352888768,0.0,1.781674498987395,0.0,0.00078125,0.0,2560,2560
logit_margin,0.0003877340233885418,1.3734117352888768,0.0,1.781674498987395,0.0,0.00078125,0.0,2560,2560
max_prob,0.0015232057427055912,1.3734117352888768,0.0,1.781674498987395,0.0,0.00078125,0.0,2560,2560
ref_attack budget=1024,0.007688692584633783,1.3734117352888768,0.0,1.781674498987395,0.0,0.00078125,0.0,2560,2560
ref_attack budget=512,-0.01587407956831166,0.6915458991929715,0.0,0.8971160372473946,0.0,0.00625,0.003125,1280,1280
LiRA K=8 budget=256,-0.2406911753971011,0.6867266127570226,0.0,0.890864161334568,0.0,0.0015625,0.0,640,640
LiRA K=8 budget=512,-0.2529265671575995,0.6802645547289315